# Quantum Funding Chart — Smooth World Edition (v3)

Fixed SVG path parser. Gray world silhouette background + geographic bubbles.

**Run all cells from the top.**

In [ ]:
from pathlib import Path
import os, subprocess

PROJECT_DIR = Path("/content/mit-prox-rd")
REPO_URL = "https://github.com/qimono-repos/mit-prox-rd.git"

if not (PROJECT_DIR / ".git").exists():
    subprocess.run(["git", "clone", "--branch", "trunk", "--single-branch", REPO_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "fetch", "origin", "trunk"], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", "-B", "trunk", "FETCH_HEAD"], check=True)

os.chdir(PROJECT_DIR)
print("Ready:", Path.cwd())

## 1. Imports and Data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, PathPatch
from matplotlib.path import Path as MplPath
import numpy as np
from pathlib import Path
import re

df = pd.read_csv("assets/quantum_funding_updated.csv", encoding="utf-8-sig")
print(f"Loaded {len(df)} records.")
display(df.head())

## 2. Configuration

In [ ]:
values = {
    "United States": 1006, "Canada": 38.5, "Brazil": 5.5,
    "United Kingdom": 330, "Germany": 848, "Netherlands": 85,
    "Switzerland": 23, "Austria": 24, "Italy": 36,
    "India": 90, "South Korea": 140, "Japan": 241,
    "Singapore": 44, "Australia": None, "China": 15000,
}

pos = {
    "Canada": (-105, 57), "United States": (-99, 37), "Brazil": (-52, -10),
    "United Kingdom": (-3, 54), "Netherlands": (5, 53), "Germany": (11, 51),
    "Switzerland": (8, 47), "Austria": (14, 47), "Italy": (12, 42),
    "India": (79, 22), "South Korea": (128, 36), "Japan": (139, 37),
    "Singapore": (104, 1.3), "Australia": (134, -25), "China": (105, 35),
}

## 3. Robust SVG World Layer

In [ ]:
def parse_svg_path(d):
    """Minimal robust parser for M / C / Z paths."""
    d = re.sub(r",", " ", d)
    d = re.sub(r"([A-Za-z])", r" \1 ", d)
    tokens = d.split()
    verts, codes = [], []
    i = 0
    while i < len(tokens):
        t = tokens[i]
        if t in ("M", "m"):
            x, y = float(tokens[i+1]), float(tokens[i+2])
            verts.append((x, y))
            codes.append(MplPath.MOVETO)
            i += 3
        elif t in ("C", "c"):
            for _ in range(3):
                x, y = float(tokens[i+1]), float(tokens[i+2])
                verts.append((x, y))
                codes.append(MplPath.CURVE4)
                i += 3
        elif t in ("Z", "z"):
            codes.append(MplPath.CLOSEPOLY)
            verts.append(verts[0] if verts else (0, 0))
            i += 1
        else:
            i += 1
    return verts, codes

def draw_world_svg(ax, svg_path="assets/world_silhouette_gray.svg"):
    import xml.etree.ElementTree as ET
    tree = ET.parse(svg_path)
    root = tree.getroot()
    paths = root.findall(".//{http://www.w3.org/2000/svg}path") or root.findall(".//path")
    for p in paths:
        d = p.get("d")
        if not d:
            continue
        verts, codes = parse_svg_path(d)
        if not verts or codes[0] != MplPath.MOVETO:
            continue
        path = MplPath(verts, codes)
        patch = PathPatch(path, facecolor="#d0d5da", edgecolor="#b8c0c8",
                          linewidth=0.7, alpha=0.82, zorder=1)
        ax.add_patch(patch)
    ax.set_xlim(-175, 185)
    ax.set_ylim(-60, 85)
    return ax

## 4. Generate Chart

In [ ]:
fig = plt.figure(figsize=(18, 11), dpi=150, facecolor="#fbfaf7")
ax = fig.add_axes([0.04, 0.16, 0.92, 0.66])
ax.set_facecolor("#fbfaf7")
ax.axis("off")

draw_world_svg(ax)

max_v = max(v for v in values.values() if v is not None and v < 5000)
for country, v in values.items():
    if country not in pos:
        continue
    x, y = pos[country]
    if v is None:
        ax.scatter([x], [y], s=80, facecolors="none", edgecolors="#6b7280", linewidths=1.5, zorder=4)
        continue
    if country == "China": :
        radius, color, edge = 6.5, "#f6e8d4", "#b96b26"
    else:
        radius = 1.6 + 7.5 * np.sqrt(v / max_v)
        color, edge = "#69c7d7", "#00a9c0"
    ax.add_patch(Circle((x, y), radius=radius, facecolor=color, edgecolor=edge,
                        linewidth=1.3, alpha=0.8, zorder=3))
    if v >= 200 and country != "China": :
        ax.text(x, y, f"{int(v)}", ha="center", va="center", fontsize=9, color="#17333a", weight="bold", zorder=5)
        ax.text(x, y + 4.5, country, ha="center", va="center", fontsize=8, color="#17333a", weight="bold", zorder=6)
    elif country == "China": :
        ax.text(x, y, "~15k", ha="center", va="center", fontsize=9, color="#7d4519", weight="bold", zorder=5)
        ax.text(x, y + 5, "China\n(cum.)", ha="center", va="center", fontsize=7.5, color="#7d4519", weight="bold", zorder=6)

fig.text(0.04, 0.92, "Quantum funding — 2026 snapshot", fontsize=22, weight="bold", color="#1a1a1a")
fig.text(0.04, 0.875, "Gray world silhouette. Bubble area scales with annual / annualized figure (USD millions).", fontsize=11, color="#444")
fig.text(0.70, 0.20, "CHINA\n≈ US$15B cumulative\nhigh uncertainty", ha="center", va="center", fontsize=8.5, color="#7d4519",
         bbox=dict(boxstyle="round,pad=0.5", facecolor="#f6e8d4", edgecolor="#b96b26"))
fig.text(0.04, 0.05, "US = FY2024 enacted. See REFERENCES.md.", fontsize=8, color="#555")
plt.show()

## Notes

- Parser fixed in v3.
- China is a cumulative estimate (not on the same scale).